# Analyzing the shell

In [1]:
import sys

import micromagneticmodel as mm
import numpy as np
import pyvista as pv

import tommos_adapter as ta

## Generate mesh

In [2]:
!tommos mesh --geom box --extent 20,20,20 --h 2 --out-name cube

[ok] Wrote data: cube.npz (nodes=3284, tets=15109)
[ok] Wrote visualization: cube.vtu (cell_data: mat_id)


## Define magnetization (0, 1, 0)

In [3]:
m_input = pv.read("cube.vtu")
m_input.points *= ta.scripts.DEFAULT_MESH_UNIT
m_input.cell_data["region_id"] = m_input.cell_data.pop("mat_id")  # Rename: `mat_id` -> `region_id`
m_input.point_data["m"] = np.tile((0.0, 1.0, 0.0), (m_input.n_points, 1))
m_input.cell_data["Ms"] = (1.76 / mm.consts.mu0) * np.ones(m_input.n_cells)  # spontaneous polarization = 1.76 Tesla
m_input

UnstructuredGrid (0x7fa4225e3040)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e-08, 1.000e-08
  Y Bounds:   -1.000e-08, 1.000e-08
  Z Bounds:   -1.000e-08, 1.000e-08
  N Arrays:   3

## Plot magnetization (needs rescailing first)

In [4]:
scaled_input = m_input.copy()
scaled_input.points /= ta.scripts.DEFAULT_MESH_UNIT

In [5]:
scaled_input.glyph(orient="m", scale=None, tolerance=0.01).plot()

2026-11-04 12:11:42.998 (   1.668s) [    7FA47D917740]vtkXOpenGLRenderWindow.:1452  WARN| bad X server connection. DISPLAY=


Widget(value='<iframe src="http://localhost:45407/index.html?ui=P_0x7fa4231b5d10_0&reconnect=auto" class="pyvi…

## Initialize System

In [6]:
system = mm.System(name="analyze_shell")
system.m = m_input.copy()

## Define energy terms

In [7]:
A = 1e-12  # exchange stiffness constant in J/m
K = 4e5  # first uniaxial anisotropy constant in J/m^3
u = (0, 0, 1)  # anisotropy unit axis
system.energy = mm.Exchange(A=A) + mm.UniaxialAnisotropy(K=K, u=u) + mm.Demag()
system.energy

Exchange(A=1e-12) + UniaxialAnisotropy(K=400000.0, u=(0, 0, 1)) + Demag()

## Drive hysteresis loop

In [8]:
hd = ta.HysteresisDriver()
hd.drive(system, h_direction=(1, 0, 0), h_start=-2 / mm.consts.mu0, h_final=1 / mm.consts.mu0, n=16)

Running Tommos (TommosRunner)[2026-11-04T12:11:43]... (13.9 s)


## Output system magnetization (with shell)

The output is not scaled anymore (real size is in nanometers) and it includes the airbox shell.

In [9]:
m_out = pv.read("analyze_shell/drive-0/hyst_analyze_shell/state_cfg00015_B+1.0000e+00T.vtu")
m_out

UnstructuredGrid (0x7fa4225e14e0)
  N Cells:    37059
  N Points:   6265
  X Bounds:   -1.139e+02, 1.139e+02
  Y Bounds:   -1.139e+02, 1.139e+02
  Z Bounds:   -1.139e+02, 1.139e+02
  N Arrays:   3

In [10]:
m_out.glyph(orient="m", scale=None, tolerance=0.001).plot()

Widget(value='<iframe src="http://localhost:45407/index.html?ui=P_0x7fa401d6c350_1&reconnect=auto" class="pyvi…

## Remove shell

We remove the shell by only selecting cells with the ID of the input mesh:

In [11]:
original_ids = np.unique(m_input["region_id"])
valid_cells = m_out.cell_data["mat_id"] == original_ids
extracted = m_out.extract_cells(valid_cells)

In [12]:
extracted.glyph(orient="m", scale=None, tolerance=0.04).plot()

Widget(value='<iframe src="http://localhost:45407/index.html?ui=P_0x7fa4028d1690_2&reconnect=auto" class="pyvi…

One arrow for each cell:

In [13]:
extracted.point_data_to_cell_data().glyph(orient="m", scale=None, tolerance=0.04).plot()

Widget(value='<iframe src="http://localhost:45407/index.html?ui=P_0x7fa401d6cdd0_3&reconnect=auto" class="pyvi…

# General comparison between `m_input` and `m_out`

In [14]:
m_input

UnstructuredGrid (0x7fa4225e3040)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e-08, 1.000e-08
  Y Bounds:   -1.000e-08, 1.000e-08
  Z Bounds:   -1.000e-08, 1.000e-08
  N Arrays:   3

In [15]:
m_out

UnstructuredGrid (0x7fa4225e14e0)
  N Cells:    37059
  N Points:   6265
  X Bounds:   -1.139e+02, 1.139e+02
  Y Bounds:   -1.139e+02, 1.139e+02
  Z Bounds:   -1.139e+02, 1.139e+02
  N Arrays:   3

In [16]:
extracted

UnstructuredGrid (0x7fa401ec32e0)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   5

`extracted` is the output mesh obtained by extracting only the cells with the original tags.

## Comparison of input and output mesh points

In [17]:
m_input.points

pyvista_ndarray([[-1.00000000e-08, -1.00000000e-08, -1.00000000e-08],
                 [ 1.00000000e-08, -1.00000000e-08, -1.00000000e-08],
                 [ 1.00000000e-08,  1.00000000e-08, -1.00000000e-08],
                 ...,
                 [-9.40025703e-09, -6.79246162e-09, -1.00000000e-08],
                 [-9.49826160e-09, -7.31540747e-09, -1.00000000e-08],
                 [-9.39096050e-09, -7.53230753e-09, -9.27043323e-09]],
                shape=(3284, 3))

In [18]:
extracted.points

pyvista_ndarray([[-10.  , -10.  , -10.  ],
                 [-10.  , -10.  ,  -8.75],
                 [-10.  , -10.  ,  -7.5 ],
                 ...,
                 [ 10.  ,  10.  ,   5.  ],
                 [ 10.  ,  10.  ,   7.5 ],
                 [ 10.  ,  10.  ,  10.  ]], shape=(3284, 3), dtype=float32)

The input mesh is scaled to real size. `scaled_input` is the input mesh rescaled to nanometers.

In [19]:
scaled_input.points

pyvista_ndarray([[-10.        , -10.        , -10.        ],
                 [ 10.        , -10.        , -10.        ],
                 [ 10.        ,  10.        , -10.        ],
                 ...,
                 [ -9.40025703,  -6.79246162, -10.        ],
                 [ -9.4982616 ,  -7.31540747, -10.        ],
                 [ -9.3909605 ,  -7.53230753,  -9.27043323]],
                shape=(3284, 3))

The (scaled) input mesh and (extracted) output mesh are shuffled:

In [20]:
scaled_input.points == extracted.points

pyvista_ndarray([[ True,  True,  True],
                 [False,  True, False],
                 [False, False, False],
                 ...,
                 [False, False, False],
                 [False, False, False],
                 [False, False, False]], shape=(3284, 3))

In [21]:
scaled_input.points[104]

pyvista_ndarray([-1.7849848 ,  0.12817606, -1.31076283])

In [22]:
extracted.points[104]

pyvista_ndarray([-10.        ,  -2.9561017 ,  -0.09847778], dtype=float32)

## On the contiguity of shell points

The shell points seem to be all appended at the end:

In [23]:
np.set_printoptions(threshold=sys.maxsize)
m_out.cell_data["mat_id"]

pyvista_ndarray([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                 1, 1, 1, 1, 1, 1, 1, 

## The cells are the same:

In [24]:
scaled_input.get_cell(204)

Cell (0x7fa401e83ee0)
  Type:        <CellType.TETRA: 10>
  Linear:      True
  Dimension:   3
  N Points:    4
  N Faces:     4
  N Edges:     6
  X Bounds:    -4.750e+00, -3.296e+00
  Y Bounds:    5.337e+00, 7.055e+00
  Z Bounds:    3.457e-01, 1.569e+00

In [25]:
m_out.get_cell(204)

Cell (0x7fa401e83e80)
  Type:        <CellType.TETRA: 10>
  Linear:      True
  Dimension:   3
  N Points:    4
  N Faces:     4
  N Edges:     6
  X Bounds:    -4.750e+00, -3.296e+00
  Y Bounds:    5.337e+00, 7.055e+00
  Z Bounds:    3.457e-01, 1.569e+00

In [26]:
extracted.get_cell(204)

Cell (0x7fa401e83ca0)
  Type:        <CellType.TETRA: 10>
  Linear:      True
  Dimension:   3
  N Points:    4
  N Faces:     4
  N Edges:     6
  X Bounds:    -4.750e+00, -3.296e+00
  Y Bounds:    5.337e+00, 7.055e+00
  Z Bounds:    3.457e-01, 1.569e+00

## The points are not the same:

In [27]:
scaled_input.points[204]

pyvista_ndarray([ 5., 10., 10.])

In [28]:
m_out.points[204]

pyvista_ndarray([-64.947914,   0.      ,  64.947914], dtype=float32)

In [29]:
extracted.points[204]

pyvista_ndarray([-10. ,   7.5,  10. ], dtype=float32)